In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
csv_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(csv_path)

print(f"Dataset shape: {df.shape}")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
# delivery_time distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop('Order_ID', axis=1)

In [ ]:
# Task 2: Write your code here:
# 2. Do we have missing values?
def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")

check_missing_values(df)


In [ ]:
# Select relevant columns, we do not select 'model' (too many unique values, too sparse and will hurt the model performance)
# Order_ID	Distance_km	Weather	Traffic_Level	Time_of_Day	Vehicle_Type	Preparation_Time_min	Courier_Experience_yrs	Delivery_Time
cols = ['Distance_km','Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type' , 'Preparation_Time_min' , 'Courier_Experience_yrs' , 'Delivery_Time']
df_clean = df[cols].copy()

# Drop rows where target (price) or key features are missing - can't predict without them
print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Weather', 'Traffic_Level', 'Time_of_Day' , 'Vehicle_Type'])
print(f"After dropping missing price/year/odometer: {df_clean.shape}")
df.head()

In [ ]:
# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day' , 'Vehicle_Type']:
    df_clean[col] = df_clean[col].fillna('unknown')

# Fill cylinders with mode - discrete feature, mode is most representative
#df_clean['cylinders'] = df_clean['cylinders'].fillna(df_clean['cylinders'].mode()[0])

print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
# 3. Do we have duplicate samples?
def check_duplicates(df):
   duplicates = df.duplicated().sum()
   print(f"Number of Duplicate Samples: {duplicates}")
   if duplicates > 0:
       print("Dropping Duplicates...")
       df.drop_duplicates(inplace=True)
       print("Duplicates Dropped.")
   else:
       print("No Duplicate Samples Found.")

check_duplicates(df)


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder
# Encode Location feature
le = LabelEncoder()
df['Weather'] = le.fit_transform(df['Weather'])
df['Traffic_Level'] = le.fit_transform(df['Traffic_Level'])
df['Time_of_Day'] = le.fit_transform(df['Time_of_Day'])
df['Vehicle_Type'] = le.fit_transform(df['Vehicle_Type'])
df.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

features = df.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 6: Write your code here:
import seaborn as sns
def check_target_imbalance(df, target_column):
   print("Target Distribution:")
   # will calculate the percentage of each class (Ex: 0.39 means 39%)
   # helps identify if one class dominates the dataset
   print(df[target_column].value_counts(normalize=True))
   sns.countplot(x=df[target_column])# Create a visual bar chart to see the distribution of classes
   plt.title("Target Distribution") #the title
   plt.show() #the final plot

check_target_imbalance(df, "Delivery_Time")


In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time",axis=1)
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

models = {
  "Ridge Regression": Ridge(alpha=1.0, max_iter=10000),
  "LASSO Regression": Lasso(alpha=1.0,  max_iter=10000),
  "Support Vector Machine": SVR(kernel='rbf'),
  "Decision Tree Regressor": DecisionTreeRegressor(max_depth=10),
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "LightGBM": LGBMRegressor(verbose=-1),
  "CatBoost": CatBoostRegressor(verbose=0)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  Create a baseline using the target median and calculate MAE score.
from sklearn.metrics import mean_absolute_error
#  using median is a robust strategy for baseline predictions
baseline_pred = df['Delivery_Time'].median()
# MAE is easier to interpret as it is in the same units as the target variable
baseline_mae = mean_absolute_error(df['Delivery_Time'], [baseline_pred] * len(df))
print(f"Baseline MAE: {baseline_mae:.4f}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
# Retrieve Ridge coefficients and sort by absolute importance
ridge_model = df["Ridge Regression"]
ridge_importance = list(zip(X.columns, ridge_model.coef_))
sorted_ridge_importance = sorted(ridge_importance, key=lambda x: abs(x[1]), reverse=True)
# Extract sorted features and their coefficients
features, coefficients = zip(*sorted_ridge_importance)
# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(features, coefficients, color='darkblue')
plt.xlabel('Coefficient Value')
plt.ylabel('Features')
plt.title('Ridge Regression Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()


In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 6))
plt.hist(valid_pred, bins=30, edgecolor='black')
plt.title('Distribution of Predictions')
plt.xlabel('Predicted Emission')
plt.ylabel('Count')
plt.show()

In [ ]:
# Task Bonus: Write your code here: